# Accent-Aware Kiswahili Speech Classification Prototype

**Project:** Raspberry Pi based Kiswahili accent classification system  
**Accent groups:** Nairobi, Coastal, Upcountry  
**Presentation date:** 18 May 2026  

This notebook is prepared for the project supervisor presentation. It summarizes the motivation, hardware/software integration, dataset, model training, current results, live demo workflow, limitations, and next steps.

## 1. Problem Statement

Kiswahili speakers pronounce words differently depending on regional background. A speech system that ignores accent variation may perform poorly for some speaker groups.

This project builds a small **accent-aware speech classification prototype** that records a speaker through a Raspberry Pi USB microphone and classifies the speaker into one of three accent groups:

- **Nairobi**
- **Coastal**
- **Upcountry**

## 2. Project Objective

The objective is to demonstrate a working hardware and software prototype that can:

1. Record speech from a USB microphone connected to a Raspberry Pi.
2. Extract speech features from the audio.
3. Classify the speaker accent group.
4. Display the predicted accent and model confidence on a local web interface.
5. Use LED and buzzer feedback during the recording process.

## 3. System Architecture

```text
Speaker
  |
  v
USB Microphone connected to Raspberry Pi
  |
  v
Streamlit Web UI / Hardware Runner
  |
  v
Audio preprocessing
- Resample to 16 kHz
- Noise reduction
- Silence trimming
- Amplitude normalization
  |
  v
MFCC feature extraction
- MFCC
- Delta
- Delta-delta
  |
  v
SVM accent classifier
  |
  v
Prediction: Nairobi / Coastal / Upcountry
  |
  v
Web UI result + confidence percentage
```

## 4. Hardware Used

| Component | Purpose |
|---|---|
| Raspberry Pi 4 | Runs the web UI, records audio, and performs classification |
| USB microphone | Captures speaker audio |
| LED | Turns on while recording is active |
| Buzzer | Sounds before recording starts and when recording ends |
| Push button / UI button | Triggers recording |

The web interface is locally deployed on the Raspberry Pi and auto-starts on boot using a `systemd` service.

## 5. Current Local Deployment

The web UI runs locally on the Raspberry Pi using Streamlit.

Current URL on the project network:

```text
http://192.168.100.139:8501
```

If the Raspberry Pi joins a different network, get the new IP address with:

```bash
hostname -I
```

Then open:

```text
http://NEW_IP:8501
```

The Streamlit app is configured to auto-start using:

```bash
sudo systemctl status srufm-streamlit.service
```

## 6. Dataset Strategy

We used a hybrid dataset approach:

| Accent group | Data source |
|---|---|
| Nairobi | Our own USB microphone recordings |
| Coastal | Our own recordings + Common Voice Kimvita support |
| Upcountry | Common Voice Kiswahili cha Bara ya Kenya support |

This allows the prototype to work while we continue collecting more real local speakers.

In [ ]:
# Run this cell from speech_recognition_project/ on the Raspberry Pi.
# It prints the current dataset summary used for training.

from pathlib import Path
import pandas as pd

metadata_path = Path("data/hybrid_accent_metadata.csv")
if not metadata_path.exists():
    raise FileNotFoundError("data/hybrid_accent_metadata.csv was not found. Run the metadata build script first.")

df = pd.read_csv(metadata_path)
print("Total rows:", len(df))
print("\nRows per accent:")
print(df["accent_label"].value_counts())
print("\nSpeakers per accent:")
print(df.groupby("accent_label")["speaker_id"].nunique())
print("\nTrain/test split:")
print(df.groupby(["accent_label", "split"]).size())

## 7. Latest Dataset Summary

After adding more Nairobi recordings, the latest hybrid dataset contains:

| Accent | Rows | Speaker count / source |
|---|---:|---|
| Nairobi | 254 | 6 local speakers |
| Coastal | 124 | 3 local speakers + Common Voice support |
| Upcountry | 120 | Common Voice support |
| **Total** | **498** | Hybrid dataset |

The dataset is now more balanced than before, but more real upcountry and coastal speakers are still needed.

## 8. Feature Extraction

The system uses **MFCC features** because they are widely used in speech processing and work well on small CPU-only projects.

Feature pipeline:

1. Load audio.
2. Convert to mono.
3. Resample to 16 kHz.
4. Reduce noise.
5. Trim silence.
6. Normalize amplitude.
7. Extract MFCC + delta + delta-delta features.
8. Feed features into the classifier.

In [ ]:
# Optional: inspect one sample and extract MFCC features.
# Run from speech_recognition_project/.

from pathlib import Path
import pandas as pd
from src.utils.audio import load_audio
from src.preprocessing.normalization import AudioNormalizer
from src.features.mfcc_extraction import FeatureExtractor

metadata = pd.read_csv("data/hybrid_accent_metadata.csv")
sample = metadata[metadata["accent_label"].eq("nairobi")].iloc[-1]
audio_path = Path("data/raw") / sample["file_path"]
print("Sample file:", audio_path)
print("Accent:", sample["accent_label"])
print("Speaker:", sample["speaker_id"])

audio, sr = load_audio(audio_path)
audio = AudioNormalizer().resample(audio, sr)
features = FeatureExtractor().extract(audio)
print("Feature vector shape:", features.shape)
print("First 10 feature values:", features[:10])

## 9. Model Used

The current model is an **SVM classifier** trained on MFCC-based audio features.

Why SVM?

- Works well on small and medium datasets.
- Does not require a GPU.
- Easy to train and deploy on Raspberry Pi.
- Gives probability/confidence estimates for predictions.

Saved model artifacts:

```text
models/accent_svm_model.joblib
models/accent_scaler.joblib
models/accent_label_encoder.joblib
```

## 10. Latest Model Evaluation

After integrating the new Nairobi recordings, the latest model produced:

| Metric | Value |
|---|---:|
| Overall accuracy | 82.76% |
| Precision | 79.66% |
| Recall | 79.24% |
| F1 score | 79.35% |

Per-accent accuracy:

| Accent | Accuracy |
|---|---:|
| Nairobi | 76.19% |
| Coastal | 61.54% |
| Upcountry | 100.00% |

Interpretation:

- Nairobi improved after adding more speakers.
- Upcountry is high because the current test data comes from Common Voice support data.
- Coastal needs more real local speakers to improve robustness.

In [ ]:
# Run a prediction on a strong Nairobi sample using the trained model.
# Run from speech_recognition_project/.

from pathlib import Path
import joblib
import pandas as pd
import numpy as np

from src.inference.predict import InferenceEngine
from src.models.svm_model import SVMModel
from src.utils.audio import load_audio
from src.preprocessing.normalization import AudioNormalizer

model = SVMModel().load("models/accent_svm_model.joblib")
scaler = joblib.load("models/accent_scaler.joblib")
encoder = joblib.load("models/accent_label_encoder.joblib")
engine = InferenceEngine(model=model, scaler=scaler, label_encoder=encoder, silence_threshold_db=55.0)

metadata = pd.read_csv("data/hybrid_accent_metadata.csv")
normalizer = AudioNormalizer()

# Pick the loudest Nairobi sample to avoid old quiet clips.
best = None
for _, row in metadata[metadata["accent_label"].eq("nairobi")].iterrows():
    path = Path("data/raw") / row["file_path"]
    try:
        audio, sr = load_audio(path)
        audio = normalizer.resample(audio, sr)
        rms_db = 20 * np.log10(np.sqrt(np.mean(audio ** 2)) + 1e-9)
        if best is None or rms_db > best[0]:
            best = (rms_db, path)
    except Exception:
        pass

print("Selected sample:", best[1])
print("RMS dB:", round(best[0], 2))
result = engine.predict_from_file(str(best[1]))

if result.is_error:
    print("Prediction error:", result.error)
else:
    print("Predicted accent:", result.predicted_word)
    print("Confidence:", f"{result.confidence * 100:.2f}%")
    print("Top predictions:")
    for label, probability in result.top_k:
        print(f"  {label}: {probability * 100:.2f}%")

## 11. Live Demonstration Procedure

Use this exact sequence during the supervisor presentation:

1. Power the Raspberry Pi and connect the USB microphone.
2. Ensure the laptop/phone is on the same network as the Pi.
3. Open the UI:

```text
http://192.168.100.139:8501
```

4. Go to the **Demo** tab.
5. Keep microphone device index as `1`.
6. Click **Record From Raspberry Pi USB Mic**.
7. Speak clearly into the Pi microphone.
8. Observe hardware feedback:
   - LED turns on during recording.
   - Buzzer sounds before and after recording.
9. Show the predicted accent and confidence percentage.
10. Click **Next Speaker / Clear Result** before the next speaker.

## 12. Important Demo Commands

Check service status:

```bash
sudo systemctl status srufm-streamlit.service
```

Restart the UI:

```bash
sudo systemctl restart srufm-streamlit.service
```

Find the Pi IP address:

```bash
hostname -I
```

Test USB microphone device detection:

```bash
cd /home/kiswahili-pi/project-srufm
python3 hardware/main.py --list-devices
```

Expected microphone:

```text
1: CMTECK: USB Audio
```

## 13. Recording More Speakers

To record a new Nairobi speaker:

```bash
cd /home/kiswahili-pi/project-srufm
python3 hardware/collect_accent_samples.py --accent nairobi --speaker-id nairobi_007 --device-index 1
```

To record a new Coastal speaker:

```bash
python3 hardware/collect_accent_samples.py --accent coastal --speaker-id coastal_004 --device-index 1
```

To record a new Upcountry speaker:

```bash
python3 hardware/collect_accent_samples.py --accent upcountry --speaker-id upcountry_001 --device-index 1
```

Recording advice:

- Speak clearly.
- Stay close to the USB microphone.
- Avoid noisy rooms.
- Do not rush the word.
- Use several speakers per accent group.

## 14. Retraining After New Recordings

After recording new speakers, rebuild the metadata and retrain:

```bash
cd /home/kiswahili-pi/project-srufm/speech_recognition_project

python3 scripts/build_hybrid_accent_metadata.py \
  --local-metadata data/accent_metadata.csv \
  --extra-metadata data/common_voice_accent_metadata.csv \
  --data-dir data/raw \
  --output data/hybrid_accent_metadata.csv

python3 main.py train \
  --target accent \
  --model svm \
  --data-dir data/raw \
  --metadata data/hybrid_accent_metadata.csv \
  --save-dir models

sudo systemctl restart srufm-streamlit.service
```

## 15. Current Limitations

This is a working prototype, but it still has limitations:

1. The dataset is still small.
2. Upcountry needs real local speaker recordings.
3. Coastal accuracy needs more speakers and cleaner recordings.
4. Some old recordings are too short or too quiet and are skipped during training.
5. The model works best for short isolated-word speech, not long continuous speech.
6. Accuracy may reduce in noisy environments.

## 16. Next Improvements

Recommended next steps:

1. Record at least 5 to 10 speakers per accent group.
2. Add real upcountry recordings.
3. Add a prediction history table to the UI.
4. Add a retrain button inside the UI.
5. Improve audio quality checks during recording.
6. Compare SVM with a neural network model after the dataset grows.
7. Package the Pi hardware in a stable case for demonstration and field testing.

## 17. Short Presentation Script

> This project is an accent-aware Kiswahili speech classification prototype. A speaker talks into a USB microphone connected to a Raspberry Pi. The system records the speech, preprocesses the audio, extracts MFCC features, and uses an SVM classifier to classify the speaker as Nairobi, Coastal, or Upcountry. The result is shown on a local web interface with a confidence percentage. The LED and buzzer provide hardware feedback during recording. The current prototype reaches about 82.76% overall test accuracy after adding more Nairobi recordings, and we expect further improvement after collecting more balanced real speaker data, especially for upcountry and coastal accents.